In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [2]:
with open("blood_work.txt","r") as f:
    blood_report = f.read()
print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
------------------------------
Hemoglobin:     15.1 g/dL         (Normal: 13.5-17.5)
Hematocrit:     44%             


In [3]:
extraction_prompt = f"""
You are a medical data extraction assistant.
From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL
based on the reference range provided in the report.
Format your response as:
- Test Name: value | Status : HIGH/LOW/NORMAL | Reference: range
Blood Report
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_value = extraction_response.text
print("=== STAGE 1 : EXTRACTED VALUES ===")

=== STAGE 1 : EXTRACTED VALUES ===


NameError: name 'extracted_values' is not defined

In [4]:
print(extracted_value)

- Hemoglobin: 15.1 g/dL | Status : NORMAL | Reference: 13.5-17.5
- Hematocrit: 44% | Status : NORMAL | Reference: 41-53%
- WBC: 6.8 x10^3/uL | Status : NORMAL | Reference: 4.5-11.0
- Platelets: 220 x10^3/uL | Status : NORMAL | Reference: 150-400
- Total Cholesterol: 238 mg/dL | Status : HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status : HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status : LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status : HIGH | Reference: <150
- Glucose (Fasting): 92 mg/dL | Status : NORMAL | Reference: 70-99
- HbA1c: 5.3% | Status : NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status : NORMAL | Reference: 0.7-1.3
- eGFR: 82 mL/min | Status : NORMAL | Reference: >60
- ALT: 28 U/L | Status : NORMAL | Reference: 7-40
- AST: 25 U/L | Status : NORMAL | Reference: 10-40
- Bilirubin Total: 0.8 mg/dL | Status : NORMAL | Reference: 0.2-1.2


In [5]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.
Based on the blood work analysis below , write:
1. A short health summary in 3 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) foods to avoid (2) Foods to eat more of.
Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_value}
"""

diet_response = llm.invoke(diet_prompt)
print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
**Health Summary**
Your blood counts, blood sugar, liver, and kidney functions are all completely normal and healthy. 
However, your lipid profile shows high "bad" cholesterol (LDL), elevated triglycerides, and low "good" cholesterol (HDL). 
This combination, known as dyslipidemia, increases cardiovascular risk and needs to be addressed primarily through dietary and lifestyle changes.

***

**Indian Diet Plan**

### 1. Foods to Avoid
* **Deep-Fried & Processed Snacks:** Samosas, pakoras, puris, bhujia, namkeen, bakery items (biscuits, patties), and foods made with refined flour (*maida*).
* **Unhealthy Fats:** *Ghee* (limit to less than 1 tsp/day), butter, *vanaspati*, palm oil, coconut oil, re-used cooking oils, and heavy cream (*malai*).
* **High-Fat Dairy & Proteins:** Full-fat milk, full-fat *paneer*, processed cheese, organ meats, red meat (mutton), and fried meats/fish.
* **Refined Sugars & Simple Carbs:** Indian sweets (*mithai*), whit

## This is a Workflow (prompt->LLM1 (Extract Numbers) -> Extracted Numbers -> LLM2 (Generate diet plan) -> diet plan

### Here LLM is not doing calls by itself